In [0]:
%sql

CREATE CATALOG IF NOT EXISTS medalhao_rocket;

In [0]:
%sql
USE CATALOG medalhao_rocket;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
USE SCHEMA default; 
CREATE VOLUME IF NOT EXISTS landing

In [0]:
from pyspark.sql import functions as F

# Definição de variáveis de caminho e import
catalogo = "medalhao_rocket"
bronze_db_name = "bronze"

In [0]:
# Definição de uma função de ingestão de dados
# A função já realiza a estruturação correta em tabelas

def ingest_csv(nome_arquivo, nome_tabela):
   
    try:
        table_name = nome_tabela
        landing_path = f"/Volumes/medalhao_rocket/default/landing/{nome_arquivo}"

        # Leitura do arquivo CSV
        # Importante notar a adição do parâmetro multiLine = True para devida importação de dados da tabela de tb_order_reviews
        # Sem eles os dados da coluna de review comment message vazavam para outras colunas
        df = spark.read.csv(landing_path, header=True, inferSchema=True, multiLine=True, escape='"')

        # Validação: arquivo vazio
        if df.count() == 0:
            raise ValueError(f"O arquivo {nome_arquivo} está vazio ou não pôde ser lido.")

        # Adiciona timestamp de ingestão
        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        # Escrita no formato Delta
        df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.{table_name}")

        print(f"Tabela bronze.{nome_tabela} criada com sucesso!\n")

    except Exception as e:
        print(f"Erro ao processar {nome_tabela}: {str(e)}")

In [0]:
# Ingerindo arquivos CSV que serão utilizados na atividade e mapeando como tabelas na layer bronze

ingest_csv('olist_customers_dataset.csv', 'tb_customers')
ingest_csv('olist_geolocation_dataset.csv', 'tb_geolocation')
ingest_csv('olist_order_items_dataset.csv', 'tb_order_items')
ingest_csv('olist_order_payments_dataset.csv', 'tb_order_payments')
ingest_csv('olist_order_reviews_dataset.csv', 'tb_order_reviews')
ingest_csv('olist_orders_dataset.csv', 'tb_orders')
ingest_csv('olist_products_dataset.csv', 'tb_products')
ingest_csv('olist_sellers_dataset.csv', 'tb_sellers')
ingest_csv('product_category_name_translation.csv', 'tb_product_category_name_translation')

In [0]:
import requests

df = spark.table('medalhao_rocket.bronze.tb_orders')

# Encontrando os valores que deverão ser utilizados no chamado da API (No caso mínimos e máximos da coluna de order_purchase_timestamp)
dates = df.select(
    F.min("order_purchase_timestamp").alias("minimo"),
    F.max("order_purchase_timestamp").alias("maximo")
).first()

valor_min = dates["minimo"]
valor_max = dates["maximo"]

# Transformando os valores para o formato correto MM-DD-YYYY
data_inicio_formatada = valor_min.strftime("%m-%d-%Y")
data_fim_formatada = valor_max.strftime("%m-%d-%Y")

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url=url)

data = response.json()
cotacoes = data["value"]

# Criando um dataframe da resposta obtida e salvando na bronze
cotacoes_df = spark.createDataFrame(cotacoes)
cotacoes_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.tb_cotacoes")

# Exibindo o dataframe para observação (é identificado que existe um gap nos dados aos finais de semana)
cotacoes_df.show()